# AI-OV7 — generated counterparts for Open Images V7 reals

`docs/02-open-weight-generators-on-open-images.md` asks for a licence-clean,
modern-era generated class whose *real* counterpart is Open Images V7. This
notebook is the generator side of it, and it makes one commitment the doc does
not: every fake is made **from a specific real**, and carries that real's
`ImageID`.

## Why pairing, and why it is not the same as "balanced"

An unpaired corpus lets a detector learn anything that happens to differ
between the two piles — camera, encoder, aspect ratio, subject matter. This
project has already measured that happening (`docs/low_level_confounds.md`).
Balancing counts does not fix it; only holding the *content* fixed does.

Here, for each fake:

* **the scene is the real's scene** — every method below is conditioned on the
  real image or on a description of it;
* **the pixel dimensions are the real's**, reached by centre-cropping to a
  multiple of 8. Nothing is ever resized, on either side, so neither class
  carries a resampling signature the other lacks;
* **the JPEG encoder is the real's own** — its 64-entry quantisation tables and
  chroma subsampling are lifted out of the source file and the fake is written
  through them. Not "a similar quality": the same integers.

That last one is not optional here. §1 of the doc is blunt that the OV7 reals
are `Thumbnail300KURL` **re-encodes**, and that saving fakes with a different
compression history would let a detector hit high accuracy on compression
alone. Per-image table copying is the strongest available answer.

## The methods, ordered by how much of the real survives

| method | conditioned on | prompt? | needs |
| --- | --- | --- | --- |
| `vae_recon` | the image | no | any latent model |
| **`self_cond`** | the image, via an **all-zero mask** | **no** | a 9-channel inpaint UNet |
| **`ref_image`** | the image, as a **reference token stream** | no | FLUX.2 klein |
| `inpaint_box` | the image + a deterministic rectangle | no | a 9-channel inpaint UNet |
| `img2img` | the image + noise at `strength` | optional | any latent model |
| `t2i_caption` | a Localized Narrative | yes | narratives on disk |

`self_cond` is the one to start with. An inpainting checkpoint's UNet takes
nine input channels — 4 noisy latent, **1 mask**, **4 latent of the masked
image**. Give it a mask of all zeros and you have said "nothing is protected",
so the reference image arrives in full as conditioning and the denoiser's task
becomes *regenerate this*. No caption is involved and none is wanted. B-Free
(CVPR 2025) built 309k fakes over 51k reals this way for exactly the bias
reason above; DRCT (ICML 2024) and TwinSynths (WACV 2025) are the same idea.

`ref_image` is the same trick for a modern flow-matching DiT, which has no mask
channels: FLUX.2 klein takes reference images directly and `Flux2KleinKVPipeline`
caches their attention K/V after the first step. It is the only **Apache-2.0**
route to image-conditioned generation — see section 3 for why that
does not mean you can run it here.

`t2i_caption` is in the mix because §3.1 wants a fully-synthetic class too, and
because it is the arm that resembles what an adversary actually does. Open
Images ships **Localized Narratives** keyed by `ImageID`, so it needs no
captioning model and no prompt-writing from you — §3.2 is explicit that
inventing captions is the wrong move.

---

# READ THIS FIRST

**What this makes.** For each real photograph, one AI counterpart of the *same
scene*, at the *same pixel size*, written through the *same JPEG quantisation
tables*. Content, geometry and encoder are held fixed, so what is left between
a pair is the generator. 1 real : 1 fake — no real is used twice.

## Running it

1. **Attach the reals.** A Kaggle Dataset containing `portrait/*.jpg` and
   `attribution.csv` from `scripts/acquire_open_images_portrait.py`.
2. **Leave `SMOKE = True`.** Run all cells. Takes minutes.
3. **Look at section 9.** Real on top, fake below. `self_cond` and `ref_image`
   must be the same scene, visibly redrawn. If a fake is a different scene, or
   identical to its real, stop and read that section's notes.
4. **Read section 12.** It prints measured seconds/image per family and the
   `N_SHARDS` that gives you a 1-hour session.
5. **Set `SMOKE = False`**, set `N_SHARDS`, run a ~2,000-image pass.
6. **Read section 10 — the gate.** It can cancel the task, and that is the
   point. Do not generate 58,000 more until it passes.

## Running it with other people

Everyone uses the same `SUITE`, `SEED`, `N_TOTAL`, `N_SHARDS`. Each person
changes **only `SHARD`**.

* `SHARD` splits work across **identical** machines. Shard *k* owns block *k*
  of the reals; blocks are disjoint, so nobody collides.
* `FAMILIES` splits work across **different** machines. A GPU with bf16 takes
  the families the others cannot run.
* **Two rules:** `N_SHARDS` identical everywhere, and no two people on the same
  `SHARD`. Break either and some reals are made twice and others never — which
  shows up as an undersized corpus, not as an error.

Nothing else is set by hand. bf16 and VRAM are detected; a family this GPU
cannot run is skipped and its share is redistributed inside the block.

## Budget, from the prior table (section 12 replaces these with measurement)

| | T4 | RTX 4060 Ti |
| --- | --- | --- |
| full 60k, with FLUX.1-schnell | ~357 h → `N_SHARDS = 431` | ~125 h |
| full 60k, without it | ~107 h → `N_SHARDS = 160` | ~38 h |
| 2,000-image gate run | 14 h / 5 h | 5 h / 2 h |

Kaggle allows 30 GPU-h per account per week. **With FLUX.1-schnell the 60k is
~2.4 weeks for five friends; without it, under one week.** It is 70% of the
budget for 20% of the images, because 12B forces CPU offload on a 16 GB card.
Decide after step 4, on measured numbers.

---

## 0. Get the code

This notebook runs in three places — Kaggle, a Colab/cloud box, and your own
machine — so nothing below hard-codes `/kaggle`. `HERE` is resolved once and
every other path hangs off it.

The repo is cloned for one reason that matters: **section 10's gate must use the
project's own proxy implementations.** The baselines it compares against
(0.5532 / 0.6721 / 0.6374) were produced by `aigcdet.features.proxies`, so a
re-implementation that is merely *equivalent* would still be comparing numbers
from two different estimators. `estimate_jpeg_quality` in particular reads the
DQT out of a JPEG and only falls back to a blockiness estimate otherwise, and
that fallback is documented as ordinal-not-calibrated — reproducing that from
memory is how a gate quietly stops meaning what the doc says it means.

If the clone fails (no network, private repo) the notebook keeps working: the
proxies are ported inline as a fallback and the gate says which one it used.

In [ ]:
import os, subprocess, sys, importlib

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"   # acquire_open_images_portrait.py and
                                          # the proxies live here, not on master

ON_KAGGLE = os.path.isdir("/kaggle/input")
HERE      = "/kaggle/working" if ON_KAGGLE else os.path.abspath("./aigc_work")
REPO_DIR  = os.path.join(HERE, "robust-aigc-detection")
os.makedirs(HERE, exist_ok=True)

def sh(argv, **kw):
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

HAVE_REPO = False
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
        sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
    else:
        sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
    for sub in ("src", "notebooks"):
        pth = os.path.join(REPO_DIR, sub)
        if pth not in sys.path:
            sys.path.insert(0, pth)
    importlib.invalidate_caches()
    sh(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
    HAVE_REPO = True
except Exception as exc:
    print(f"clone failed ({type(exc).__name__}: {exc}).")
    print("Continuing standalone -- section 10 will use the ported proxies and "
          "will say so. Everything else is self-contained.")

print(f"\\nrunning on {'Kaggle' if ON_KAGGLE else 'a local/cloud box'}; work dir {HERE}")

## 1. Parameters

In [ ]:
# =================== THE ONLY LINES YOU NORMALLY EDIT ===================
SMOKE     = True    # 3 images per family, every family, one session. Do this
                    # first, LOOK at section 8, read section 10, and only then
                    # set False. docs/02 §4: "Start with 2,000 total and stop."
N_TOTAL   = 2000    # total fakes when SMOKE is False, split across the suite
SHARD, N_SHARDS = 0, 1
# =======================================================================

SMOKE_PER_FAMILY = 3
SEED = 20260830

# ---- shard budget ----------------------------------------------------------
TARGET_SHARD_HOURS = 1.0   # what ONE session of this notebook should cost.
                           # Section 12 turns measured throughput into the
                           # N_TOTAL / N_SHARDS that hits this, per device.
FAMILIES = None            # None = every family this GPU can run.
                           # Or a list of family names, to hand ONE machine a
                           # specific job: FAMILIES = ["flux2_klein9bkv_ref_image"]
                           # on the box with bf16, while the T4s take the rest.
                           # This is the axis to split heterogeneous machines on.
                           # SHARD splits one workload across IDENTICAL machines;
                           # FAMILIES splits it across DIFFERENT ones. Using
                           # SHARD alone for a mixed fleet makes two machines
                           # regenerate the same SD rows to reach the one family
                           # only one of them can do.
STRICT_SUITE = False       # False: a family this GPU cannot run is SKIPPED with
                           # a reason, so one notebook runs everywhere. True: it
                           # raises instead -- use when a machine is supposed to
                           # be covering specific families and silence would
                           # mean an under-filled corpus nobody noticed.

#: Prior throughput in seconds/image, at ~0.30 MP, on a **T4**. These are
#: ESTIMATES and are used for nothing but the shard plan. Section 12 overwrites
#: them from the smoke run's real numbers and writes them to disk, so the second
#: session on a machine plans from measurement rather than from this table.
RATE_PRIOR_T4 = {
    ("sd15_inpaint",  "self_cond"):   5.0,
    ("sd21_inpaint",  "self_cond"):   7.0,
    ("sd21_inpaint",  "inpaint_box"): 7.0,
    ("sd21_inpaint",  "vae_recon"):   0.3,
    ("sdxl_inpaint",  "self_cond"):  14.0,
    ("sdxl_inpaint",  "inpaint_box"): 14.0,
    ("sdxl_base",     "t2i_caption"): 16.0,
    ("flux_schnell",  "t2i_caption"): 90.0,   # 12B through CPU offload on 16 GB
    ("z_image_turbo", "t2i_caption"): 8.0,
    ("flux2_klein_4b",    "ref_image"): 25.0,
    ("flux2_klein_9b_kv", "ref_image"): 12.0,
}
#: Rough speed relative to a T4, matched on the device name. Only ever scales
#: the ESTIMATE; a measured rate ignores it entirely.
DEVICE_FACTOR = {"T4": 1.0, "P100": 1.2, "V100": 0.6, "L4": 0.45, "A10": 0.4,
                 "4060": 0.35, "4070": 0.28, "4080": 0.2, "4090": 0.15,
                 "A100": 0.2, "H100": 0.1, "3090": 0.25, "5090": 0.1}

# The suite. One row per FAMILY, and a family is (model, method) -- docs/02 §3.3
# is explicit that `flux_schnell_t2i` and `flux_schnell_inpaint` are two
# families and not one, because the held-out design is built on family names
# meaning something. `share` is this family's TARGET fraction; section 5 renormalises it over
# the families the running GPU can actually do.
#
# The shape below is ~70% fully synthetic / 30% partially synthetic (§3.1) over
# three DECODER lineages (§3.4). Comment rows out rather than deleting them;
# the shares are renormalised, so a commented row costs nothing.
SUITE = [
    dict(model="sd21_inpaint",  method="self_cond",   share=0.25),
    dict(model="sdxl_inpaint",  method="self_cond",   share=0.25),
    dict(model="flux_schnell",  method="img2img",     share=0.20),
    dict(model="sd21_inpaint",  method="inpaint_box", share=0.15),
    dict(model="sdxl_inpaint",  method="inpaint_box", share=0.10),
    dict(model="sd21_inpaint",  method="vae_recon",   share=0.05),
    # Apache-2.0 and a fourth decoder lineage. Needs bf16, so a T4 skips it and
    # the shard rebalances over the five that remain; a 40-series card runs all
    # six. NOTE 4B, not 9B: the 9B kleins are FLUX Non-Commercial v2.1 and the
    # registry refuses them.
    dict(model="flux2_klein_4b", method="ref_image",   share=0.20),
]

PAIRING = "disjoint"
#   "disjoint" -> each family draws its OWN slice of reals, so the corpus comes
#                 out 1 real : 1 fake. docs/02 §4 asks for exactly this ("60,000
#                 reals means ~60,000 fakes"), and it is what you want for
#                 TRAINING: six fakes of one scene is six correlated rows, not
#                 six independent ones.
#   "shared"   -> every family draws the SAME reals, so one scene gets one fake
#                 per generator. Use for an ABLATION that compares generators on
#                 identical content. It is a 1:N imbalance by construction.

CORPUS = "open_images_v7"     # "open_images_v7" | "coco"
COCO_SPLIT = "train2017"      # only read when CORPUS == "coco"
I_KNOW_VAL2017_IS_THE_BENCHMARK = False

# ---- geometry: no resampling, ever -----------------------------------------
MIN_SIDE = 320    # skip reals below this; OV7 thumbnails sit at p05 ~360
MAX_SIDE = 1024   # cap the long side by CROPPING, never by resizing. A 4-step
                  # model at 1536px is slow and off-distribution; a crop costs
                  # field of view and costs no spectral fidelity.

# ---- encoder parity (doc §5.2) ---------------------------------------------
COPY_QTABLES = True
FAKE_ENCODE_PASSES = 1   # raise to 2 only if section 9 says compression still
                         # separates the classes

# ---- output -----------------------------------------------------------------
SOURCE   = "open_images_v7" if CORPUS == "open_images_v7" else "coco_pairs"
OUT_ROOT = os.path.join(HERE, "aigc_pairs")
OUT_ROOT = None   # resolved in section 0 to <HERE>/aigc_pairs
EMIT_REALS = True   # copy the paired reals in BYTE-FOR-BYTE. Re-encoding them
                    # to "match" would add a compression generation to the
                    # authentic class and manufacture the very confound the
                    # qtable copy exists to remove.

In [ ]:
# ---------------------------------------------------------------------------
# Model registry.
#
# `licence` is the WEIGHT licence, read at the model page. docs/02 §2 is the
# authority on which are usable, with ONE correction made here:
#
#   The doc's table reads "SDXL 1.0 / SDXL-Turbo | CreativeML OpenRAIL++-M |
#   Use." That is right for SDXL 1.0 and WRONG for SDXL-Turbo, whose card says
#   `sai-nc-community` -- non-commercial, "for commercial use please refer to
#   https://stability.ai/membership". SDXL-Turbo is therefore absent from this
#   registry, and the doc's table should be fixed. FLUX.1-dev and SANA are
#   absent for the reason the doc already gives.
#
# `lineage` is the DECODER, and it is the field the held-out split is grouped
# by (§3.4): everything sharing the SD VAE leaves the same trace, so holding
# out `sdxl_*` while `sd21_*` trains measures a cousin, not a stranger.
#
# `needs_bf16` marks the flow-matching DiTs. It is not a preference: fp16 has
# 5 exponent bits against bf16's 8, and these models were trained in the wider
# range. Section 2 turns it into a hard gate, because Kaggle has no GPU that
# supports bf16 and a silent fp16 fallback produces plausible-looking noise.
# ---------------------------------------------------------------------------
MODELS = {
    "sd15_inpaint": dict(
        hf_id="stable-diffusion-v1-5/stable-diffusion-inpainting",
        licence="CreativeML OpenRAIL-M", licence_tag="creativeml-openrail-m",
        commercial=True, lineage="sd_vae", family="sd15",
        task="inpaint", methods=("self_cond", "inpaint_box", "vae_recon"),
        steps=50, guidance=7.5, strength=1.0, vram_gb=6),
    "sd21_inpaint": dict(
        hf_id="stabilityai/stable-diffusion-2-inpainting",
        licence="CreativeML OpenRAIL++-M", licence_tag="openrail++",
        commercial=True, lineage="sd_vae", family="sd21",
        task="inpaint", methods=("self_cond", "inpaint_box", "vae_recon"),
        steps=50, guidance=7.5, strength=1.0, vram_gb=8),
    "sdxl_inpaint": dict(
        hf_id="diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        licence="CreativeML OpenRAIL++-M", licence_tag="openrail++",
        commercial=True, lineage="sdxl_vae", family="sdxl",
        task="inpaint", methods=("self_cond", "inpaint_box", "vae_recon"),
        # The card is explicit that strength must stay BELOW 1.0 on this one.
        steps=25, guidance=8.0, strength=0.99, vram_gb=12),
    "sdxl_base": dict(
        hf_id="stabilityai/stable-diffusion-xl-base-1.0",
        licence="CreativeML OpenRAIL++-M", licence_tag="openrail++",
        commercial=True, lineage="sdxl_vae", family="sdxlb",
        task="t2i", methods=("img2img", "t2i_caption", "vae_recon"),
        steps=30, guidance=7.0, strength=0.75, vram_gb=12),
    # NOTE THE VERSION. This is FLUX **.1**-schnell, and it is NOT marked
    # needs_bf16: FLUX.1 is routinely run in fp16 without falling apart, so a
    # T4 RUNS it. What a T4 cannot run is FLUX **.2** klein, further down. The
    # two are unrelated for this purpose and confusing them is easy -- the
    # thing that makes .1-schnell expensive on a 16 GB card is its 12B size
    # forcing CPU offload, which is a SPEED problem, not a capability one.
    "flux_schnell": dict(
        hf_id="black-forest-labs/FLUX.1-schnell",
        licence="Apache-2.0", licence_tag="apache-2.0", commercial=True,
        lineage="flux_vae", family="flux_schnell",
        task="t2i", methods=("img2img", "t2i_caption", "vae_recon"),
        # FLUX.1 is the one DiT here that is widely run in fp16 without falling
        # apart, so it is NOT marked needs_bf16 -- but it is 12B, so on a 16 GB
        # card it runs through sequential CPU offload and it is slow.
        steps=4, guidance=0.0, strength=0.75, vram_gb=24),
    "sd35_medium": dict(
        hf_id="stabilityai/stable-diffusion-3.5-medium",
        licence="Stability Community License -- free below the revenue "
                "threshold stated on the model page. Record that threshold in "
                "the source docstring before shipping anything built on it",
        commercial=True, lineage="sd3_vae", family="sd35m",
        task="t2i", methods=("img2img", "t2i_caption", "vae_recon"),
        steps=28, guidance=4.5, strength=0.75, vram_gb=16,
        needs_bf16=True, gated=True),
    "z_image_turbo": dict(
        hf_id="Tongyi-MAI/Z-Image-Turbo",
        licence="Apache-2.0", licence_tag="apache-2.0", commercial=True,
        lineage="zimage_vae", family="z_image_turbo",
        task="t2i", methods=("img2img", "t2i_caption", "vae_recon"),
        # The card: Turbo is distilled and guidance MUST be 0.
        steps=9, guidance=0.0, strength=0.75, vram_gb=16,
        needs_bf16=True, pipe_class="ZImagePipeline"),
    # ---- FLUX.2 klein: the Apache-2.0 route to image conditioning ----------
    # Every klein takes reference images directly, so `ref_image` is the DiT
    # analogue of self_cond: no mask channels, no caption, the picture is the
    # condition. The -kv variant caches the reference tokens' attention K/V
    # after step 1, which is why its default is 4 steps rather than 50.
    # THE SIZE IS THE LICENCE, and this is the trap docs/02 §2 was written
    # about. From the klein-4B model card: "we approved the release of the
    # open-weight FLUX.2 [klein] 4B models under an Apache 2.0 license and the
    # release of the FLUX.2 [klein] 9B models under a non-commercial license".
    # So 4B and klein-base-4B are usable, and EVERY 9B -- klein-9B,
    # klein-base-9B, klein-9b-kv, and their fp8/nvfp4 repackings -- is FLUX
    # Non-Commercial v2.1 and is not, on exactly the grounds the doc already
    # rules out FLUX.1-dev. The 9B entry stays here REFUSED rather than
    # deleted: "reach for the bigger one" is the obvious move, and an absent
    # entry reads as an oversight where a refused one reads as a decision.
    "flux2_klein_4b": dict(
        hf_id="black-forest-labs/FLUX.2-klein-4B",
        licence="Apache-2.0", licence_tag="apache-2.0", commercial=True,
        lineage="flux2_vae", family="flux2_klein4b",
        task="t2i", methods=("ref_image", "t2i_caption", "vae_recon"),
        steps=28, guidance=4.0, strength=0.75, vram_gb=13,
        needs_bf16=True, gated=True, pipe_class="Flux2KleinPipeline"),
    "flux2_klein_9b_kv": dict(
        hf_id="black-forest-labs/FLUX.2-klein-9b-kv",
        licence="FLUX Non-Commercial v2.1 -- NOT USABLE, see the note above",
        commercial=False,
        lineage="flux2_vae", family="flux2_klein9bkv",
        task="t2i", methods=("ref_image", "t2i_caption", "vae_recon"),
        steps=4, guidance=None, strength=0.75, vram_gb=20,
        needs_bf16=True, gated=True, pipe_class="Flux2KleinKVPipeline"),
    "qwen_image": dict(
        hf_id="Qwen/Qwen-Image",
        licence="Apache-2.0", licence_tag="apache-2.0", commercial=True,
        lineage="qwen_vae", family="qwen_image",
        task="t2i", methods=("img2img", "t2i_caption", "vae_recon"),
        steps=30, guidance=4.0, strength=0.75, vram_gb=40, needs_bf16=True),
}

METHODS = {
    "self_cond":   ("inpaint", "empty-mask inpaint: the image IS the condition"),
    "ref_image":   ("t2i",     "FLUX.2 reference conditioning: the image IS the condition"),
    "inpaint_box": ("inpaint", "regenerate a deterministic rectangle, keep the rest"),
    "img2img":     ("img2img", "SDEdit: renoise to `strength`, denoise back"),
    "t2i_caption": ("t2i",     "text-to-image from a Localized Narrative"),
    "vae_recon":   ("t2i",     "encode/decode through the VAE only, no denoiser"),
}

def family_of(entry):
    return f"{MODELS[entry['model']]['family']}_{entry['method']}"

# --- validate the suite before anything downloads --------------------------
seen = set()
for e in SUITE:
    assert e["model"] in MODELS, f"unknown model {e['model']!r}"
    assert e["method"] in METHODS, f"unknown method {e['method']!r}"
    spec = MODELS[e["model"]]
    assert e["method"] in spec["methods"], (
        f"{e['model']} cannot do {e['method']!r}; it offers {spec['methods']}. "
        "self_cond and inpaint_box need a 9-CHANNEL inpainting checkpoint -- a "
        "base t2i UNet has no mask input for the reference image to arrive by. "
        "ref_image needs a FLUX.2 klein pipeline.")
    assert spec.get("commercial"), (
        f"{e['model']} is {spec['licence']}. docs/02 §2's rule is that the "
        "licence on the WEIGHTS governs: a non-commercial model does not "
        "become usable because its outputs are fine, or because a bigger "
        "sibling was tempting. This is the same call the doc already made "
        "against FLUX.1-dev and SANA. Remove it from SUITE.")
    f = family_of(e)
    assert f not in seen, f"family {f!r} appears twice in SUITE"
    seen.add(f)

# A planning figure only: the real per-family counts are set in section 6, from
# this machine's shard block and what this GPU can actually run.
_tot = sum(e["share"] for e in SUITE)
for e in SUITE:
    e["n"] = SMOKE_PER_FAMILY if SMOKE else max(1, round(N_TOTAL * e["share"] / _tot))

print(f"corpus   {CORPUS}   pairing={PAIRING}   smoke={SMOKE}")
print(f"{'family':28s} {'model':18s} {'lineage':11s} {'n':>6s}  licence")
for e in SUITE:
    s = MODELS[e["model"]]
    print(f"{family_of(e):28s} {e['model']:18s} {s['lineage']:11s} "
          f"{e['n']:6d}  {s['licence'][:44]}")
print(f"\ntotal {sum(e['n'] for e in SUITE)} fakes over "
      f"{len({MODELS[e['model']]['lineage'] for e in SUITE})} decoder lineages")

## 2. Install

Kaggle ships a torch built against this machine's drivers, and `diffusers`
pulls a tree deep enough that pip will happily resolve a different one into it
— which costs the session its GPU. The plan is printed before it runs. **If you
ever see `torch` in the plan, stop.**

FLUX.2 klein needs a recent `diffusers`; `Flux2KleinKVPipeline` did not exist
before it. The version floor is checked against what the suite actually asks
for, so a suite of SD families does not force an upgrade it does not need.

In [ ]:
import importlib, subprocess, sys

def sh(argv):
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True)

def version_of(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

def vt(s):
    return tuple(int(x) for x in (s or "0").split(".")[:3] if x.isdigit())

need = [d for d in ("diffusers", "transformers", "accelerate", "safetensors",
                    "sentencepiece", "protobuf", "pandas", "pyarrow")
        if version_of(d) is None]

FLOOR = (0, 27, 0)
if any(MODELS[e["model"]].get("pipe_class", "").startswith("Flux2") for e in SUITE):
    FLOOR = (0, 36, 0)     # Flux2Klein* pipelines
if vt(version_of("diffusers")) < FLOOR:
    need.append("diffusers")
if any(MODELS[e["model"]].get("pipe_class") == "ZImagePipeline" for e in SUITE):
    need.append("diffusers")

plan = [[sys.executable, "-m", "pip", "install", "-q", "--upgrade", *sorted(set(need))]] if need else []
print("pip plan:")
for c in plan or [["(nothing to install)"]]:
    print("   ", " ".join(c))
assert not any(w.split("=")[0].split(">")[0] in ("torch", "torchvision", "triton")
               for c in plan for w in c), "STOP: the plan would touch torch"
for c in plan:
    sh(c)

importlib.invalidate_caches()
import torch, diffusers
print(f"\ntorch {torch.__version__}   diffusers {diffusers.__version__}")

## 3. Hardware — what this GPU can and cannot make

Kaggle offers **T4 x2** (Turing, sm_75) and **P100** (Pascal, sm_60), 16 GB
each, 30 GPU-hours a week. Neither supports **bfloat16**.

That is a capability boundary, not a performance note. FLUX.2, SD 3.5, Z-Image and
Qwen-Image are flow-matching DiTs trained in bf16, whose 8 exponent bits they
use. fp16 has 5. Running them in fp16 does not raise — it produces washed-out
or NaN-speckled images that still look like images, which is the worst possible
failure for a corpus whose whole job is to record what a generator's output
looks like. So the check below **skips those families and says why**, rather than
falling back to fp16. Skipping is the right verb rather than refusing: the same
notebook is meant to open on Kaggle and on a 4060 Ti, and each shard's block of
reals is rebalanced (section 5) over whatever families the machine in front of
you can run, so nothing is stranded by a T4 skipping one.

There is a second reason to refuse rather than quantise. NF4 or fp8 weights
would fit, and the pictures would look fine. But you are building a corpus to
teach a detector *this generator's traces*, and a 4-bit model's traces are
partly your compute budget's. Quantise for a demo, not for training data.

**What this means in practice.** An RTX 40-series, L4, A10, A100 or newer has
bf16 and runs everything. Kaggle's T4/P100 runs the SD-family families plus
`flux_schnell` — FLUX.1 is the one DiT here routinely run in fp16 without
falling apart — and skips the rest. Both are useful; they are just covering
different parts of the same declared suite.

In [ ]:
assert torch.cuda.is_available(), (
    "no GPU. Settings > Accelerator > GPU T4 x2 (or P100), then restart.")
CAP = torch.cuda.get_device_capability(0)
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
HAS_BF16 = CAP[0] >= 8
print(f"{GPU}  sm_{CAP[0]}{CAP[1]}  {VRAM:.0f} GB  bf16={'yes' if HAS_BF16 else 'NO'}")

def cannot_run(entry):
    """Why this GPU cannot run this family, or None."""
    spec = MODELS[entry["model"]]
    if spec.get("needs_bf16") and not HAS_BF16:
        return (f"needs bfloat16; {GPU} is sm_{CAP[0]}{CAP[1]}. Not a speed "
                "issue: fp16 has 5 exponent bits against bf16's 8, and these "
                "models use the range. fp16 does not raise -- it emits "
                "washed-out or NaN-speckled images that still look like "
                "images, which for a corpus recording what a generator's "
                "output LOOKS LIKE is the worst available failure.")
    return None

if FAMILIES is not None:
    known = {family_of(e) for e in SUITE}
    unknown = set(FAMILIES) - known
    assert not unknown, f"FAMILIES names nothing in SUITE: {sorted(unknown)}"

RUNNABLE, SKIPPED = [], []
for e in SUITE:
    why = cannot_run(e)
    if not why and FAMILIES is not None and family_of(e) not in FAMILIES:
        why = "not in FAMILIES -- this machine was not assigned it"
    (SKIPPED if why else RUNNABLE).append(e)
    if why:
        e["skip_reason"] = why

print(f"\n{'family':28s} {'model':18s} {'bf16?':>6s}  status")
for e in SUITE:
    spec = MODELS[e["model"]]
    need = "yes" if spec.get("needs_bf16") else "no"
    print(f"{family_of(e):28s} {e['model']:18s} {need:>6s}  "
          f"{'RUN' if e in RUNNABLE else 'skip'}")

if SKIPPED:
    print()
    for e in SKIPPED:
        print(f"SKIP {family_of(e):26s} {e['skip_reason'][:96]}")
    print("\nThis shard's block of reals is REBALANCED over the families that "
          "remain (section 5), so no image is stranded -- the skipped families "
          "get their images from shards run on a machine that has bf16. Do not "
          "quantise them onto this GPU to fill the gap: a 4-bit model's traces "
          "are partly your compute budget's, and the corpus would record both.")
    if STRICT_SUITE:
        raise SystemExit(
            "STRICT_SUITE is on and this GPU cannot run every declared family. "
            "Either move to an Ampere-or-newer card, or set STRICT_SUITE=False "
            "and let another machine cover the skipped rows.")

assert RUNNABLE, (
    f"{GPU} can run none of the declared families. Every entry needs bf16 and "
    "this card is sm_"f"{CAP[0]}{CAP[1]}. Kaggle offers only T4 (sm_75) and "
    "P100 (sm_60); an RTX 40-series, L4, A10, A100 or newer has bf16.")

tight = [e for e in RUNNABLE if MODELS[e["model"]]["vram_gb"] > VRAM]
if tight:
    print(f"\nCPU offload (10-30x slower) for: "
          f"{sorted({e['model'] for e in tight})}")

DEV_FACTOR = next((v for k, v in DEVICE_FACTOR.items() if k in GPU), 1.0)
print(f"\n{len(RUNNABLE)} of {len(SUITE)} families runnable here  "
      f"| speed prior {DEV_FACTOR:.2f}x a T4")

# HF token. Gated repos need one; it is read from a Kaggle Secret and never
# printed. Add-ons > Secrets > new secret named HF_TOKEN.
HF_TOKEN = None
if any(MODELS[e["model"]].get("gated") for e in RUNNABLE):
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            HF_TOKEN = None
    HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not HF_TOKEN:
        try:                      # a `huggingface-cli login` on your own box
            from huggingface_hub import HfFolder
            HF_TOKEN = HfFolder.get_token()
        except Exception:
            pass
    assert HF_TOKEN, (
        "the runnable suite includes a gated model but no token was found. "
        "Accept the model's terms on its Hugging Face page while logged in, "
        "create a read token, then either: on Kaggle, Add-ons > Secrets > "
        "HF_TOKEN and attach it; elsewhere, `huggingface-cli login` or export "
        "HF_TOKEN. The token is never printed and never leaves this process.")
    print("HF token found (not printed)")

## 4. The reals

**Open Images V7.** `scripts/acquire_open_images_portrait.py` (on
`origin/feat/robust-aigc-detection`, not on `master`) leaves
`portrait/<ImageID>.jpg` plus an `attribution.csv` carrying `ImageID`, `size`,
`ratio`, `Author`, `AuthorProfileUrl`, `Title`, `OriginalURL` and `License`.
Publish that directory as a Kaggle Dataset and attach it; the CSV is read as
the source of truth, which means dimensions come from it and no file has to be
opened to build the pool.

**`attribution.csv` is a filter here, not just a record.** §5.4 requires one row
per real with no blanks in `Author` or `OriginalURL`. A real whose attribution
is incomplete is one whose CC BY obligation cannot be met, so it is dropped from
the pool rather than generated from — that way the emitted corpus satisfies
§5.4 by construction instead of failing an audit later.

**COCO** is kept as a second adapter because the machinery is identical. Note
the guard: `val2017` is the authentic half of the organisers' demo benchmark and
`sources.py` marks it `exclude_from_training=True`. A fake generated *from* a
val2017 real is content-paired with a benchmark image, so training on it leaks
the benchmark while every row-level exclusion still reads clean.

In [ ]:
import glob, hashlib, io, json, os, shutil, time
import numpy as np
import pandas as pd
from PIL import Image

#: Where to look for the reals. Kaggle mounts first, then the paths the
#: harvest script writes to on a workstation. Add your own if it lives
#: elsewhere -- this is the one thing that is genuinely machine-specific.
SEARCH_ROOTS = ["/kaggle/input", os.path.expanduser("~/data"),
                "/mnt/berstorage/techjam", "./data", HERE]

def one(pattern, what):
    hits = []
    for root in SEARCH_ROOTS:
        if os.path.isdir(root):
            hits += glob.glob(os.path.join(root, pattern), recursive=True)
    hits = sorted(set(hits))
    assert hits, f"no {what} attached (looked for {pattern})"
    assert len(hits) == 1, (
        f"{len(hits)} candidates for {what}: {hits}. Detach the ones this run "
        "is not using -- picking one silently is how a corpus ends up "
        "describing data it does not contain.")
    return hits[0]

if CORPUS == "open_images_v7":
    ATTRIB = one("**/attribution.csv", "OV7 attribution.csv")
    IMG_DIR = os.path.dirname(one("**/portrait/*.jpg",
                                  "OV7 portrait/ directory"))
    att = pd.read_csv(ATTRIB)
    print(f"attribution.csv: {len(att)} rows from {ATTRIB}")

    need_cols = {"ImageID", "size", "Author", "OriginalURL", "License"}
    assert need_cols <= set(att.columns), (
        f"attribution.csv is missing {sorted(need_cols - set(att.columns))}. "
        "This is not the file acquire_open_images_portrait.py writes.")

    # §5.4, applied as a filter rather than as a post-hoc assertion.
    ok = att["Author"].fillna("").str.strip().ne("") & \
         att["OriginalURL"].fillna("").str.strip().ne("")
    print(f"attribution complete: {int(ok.sum())} of {len(att)} "
          f"({100*ok.mean():.1f}%) -- the rest cannot meet CC BY and are dropped")
    att = att[ok]

    lic = att["License"].str.replace("https://", "").str.replace("http://", "")
    bad = att[~lic.str.startswith("creativecommons.org/licenses/by/2.0")]
    assert bad.empty, (
        f"{len(bad)} reals are not CC BY 2.0, e.g. {bad['License'].iloc[0]!r}. "
        "docs/02 §1 is explicit that CC BY 2.0 is the whole point of this "
        "source -- it is the only vertical-real licence audited that clears "
        "both commercial use and redistribution. Do not silently mix.")

    wh = att["size"].str.split("x", expand=True).astype(int)
    POOL = pd.DataFrame(dict(
        image_id=att["ImageID"].astype(str),
        file_name=att["ImageID"].astype(str) + ".jpg",
        width=wh[0], height=wh[1],
        licence=att["License"], author=att["Author"],
        original_url=att["OriginalURL"],
    ))
else:
    assert COCO_SPLIT in ("train2017", "val2017")
    assert not (COCO_SPLIT == "val2017" and not I_KNOW_VAL2017_IS_THE_BENCHMARK), (
        "val2017 is the authentic half of the organisers' demo benchmark and "
        "sources.py marks it exclude_from_training=True. A fake generated FROM "
        "a val2017 real is content-paired with a benchmark image: train on it "
        "and you have leaked the benchmark while every row-level exclusion "
        "still reads clean. Set the flag only for rows headed for --demo-dir.")
    cap = one(f"**/annotations/captions_{COCO_SPLIT}.json", "COCO captions")
    IMG_DIR = os.path.dirname(one(f"**/{COCO_SPLIT}/000000*.jpg",
                                  "COCO images"))
    with open(cap) as f:
        cj = json.load(f)
    # Per-image Flickr licences. 1/2/3 are NonCommercial and 6 is NoDerivatives
    # -- an AI counterpart is a derivative -- so only {4,5,7,8} clear both.
    OK_IDS = frozenset({4, 5, 7, 8})
    EXPECT = {1: "by-nc-sa/2.0", 2: "by-nc/2.0", 3: "by-nc-nd/2.0", 4: "by/2.0",
              5: "by-sa/2.0", 6: "by-nd/2.0", 7: "flickr.com/commons",
              8: "usa.gov/copyright"}
    for l in cj["licenses"]:
        assert EXPECT[l["id"]] in l["url"], (
            f"COCO licence id {l['id']} is now {l['url']!r}, not what OK_IDS "
            "was written against. Reclassify before generating.")
    im = pd.DataFrame(cj["images"])
    im = im[im["license"].isin(OK_IDS)]
    LICNAME = {l["id"]: l["url"] for l in cj["licenses"]}
    POOL = pd.DataFrame(dict(
        image_id=im["id"].astype(str), file_name=im["file_name"],
        width=im["width"], height=im["height"],
        licence=im["license"].map(LICNAME), author="(flickr)",
        original_url=im["flickr_url"],
    ))
    CAPTION_JSON = cj

short = POOL[["width", "height"]].min(axis=1)
POOL = POOL[short >= MIN_SIDE].reset_index(drop=True)
FILE_NAME = dict(zip(POOL["image_id"], POOL["file_name"]))
print(f"\npool after MIN_SIDE={MIN_SIDE}: {len(POOL)} reals")
print(f"short side  min {short.min()}  median {int(short.median())}")
POOL.head(3)

## 5. Who gets which reals — a block per shard, rebalanced inside it

Reals are ordered once by `blake2b(SEED, ImageID)`, and **shard *k* owns block
*k* of that order**. Blocks are contiguous and disjoint, so two people holding
different `SHARD` numbers can never touch the same photograph — whatever their
GPUs can or cannot run. That is the only cross-machine guarantee needed, and it
is the whole coordination protocol.

**Inside its block, the work is split across the families *this* machine can
run.** A T4 that skips `flux2_klein4b_ref_image` spreads a 100-image block over
five families, ~20 each. A 4060 Ti spreads the same-sized block over six, ~17
each. Nothing is stranded and nothing is duplicated: under `PAIRING="disjoint"`
no real is reused anyway, so *which* real lands in which family does not matter
— only that each is used exactly once. The cell asserts that partition holds.

**The cost of rebalancing, stated plainly.** Per-family *counts* now depend on
the fleet's mix of GPUs, so `share` becomes a target rather than a guarantee. If
only one machine in the fleet has bf16, the klein family comes out thin. Section
11 prints the counts actually achieved, and the fix is to run a few shards with
`FAMILIES` set to just the thin one.

**The one rule.** `N_SHARDS` must be identical on every machine — it is the
divisor that defines what a block *is* — and no two people may hold the same
`SHARD`. Break either and you get some reals generated twice and others never,
which surfaces as a quietly undersized corpus rather than as an error.

**`PAIRING` is a real experimental choice.** `disjoint` gives 1 real : 1 fake,
which is what §4 asks for and what you want to train on — six fakes of one scene
are six correlated rows, and a split that separates them lets the model see the
same scene under both labels. `shared` hands every family the whole block, so
one scene gets a fake from every generator: that is what you want for an
*ablation* comparing generators with content held fixed. Different corpora; pick
on purpose.

In [ ]:
def order_key(i):
    return hashlib.blake2b(f"{SEED}:{i}".encode(), digest_size=8).hexdigest()

ORDER = POOL.assign(_k=[order_key(i) for i in POOL["image_id"]]) \
            .sort_values("_k").reset_index(drop=True)

# Each SHARD owns a contiguous, disjoint BLOCK of the shuffled order. That is
# the only cross-machine guarantee needed, and it is the one that matters: two
# people holding different SHARD numbers can never touch the same real, whatever
# families their GPUs can run.
#
# WITHIN the block, the work is split across the families THIS machine can run,
# renormalised. A T4 that skips klein spreads its block over five families; a
# 40-series card spreads the same-sized block over six. Nothing is stranded and
# nobody duplicates, because under PAIRING="disjoint" no real is reused anyway,
# so which real ends up in which family does not matter -- only that each is
# used once.
#
# The cost, stated: per-family COUNTS then depend on the fleet's mix of GPUs, so
# `share` becomes a target rather than a guarantee. Section 11 prints the counts
# actually achieved; top up a thin family by running a few shards with
# FAMILIES set to just that one.
BLOCK = (SMOKE_PER_FAMILY * len(RUNNABLE)) if SMOKE else max(1, N_TOTAL // N_SHARDS)
assert len(ORDER) >= BLOCK * N_SHARDS, (
    f"{len(ORDER)} eligible reals but {N_SHARDS} shards of {BLOCK} need "
    f"{BLOCK * N_SHARDS}. Lower N_TOTAL, raise MIN_SIDE's yield, or harvest more.")

block = ORDER.iloc[SHARD * BLOCK:(SHARD + 1) * BLOCK].reset_index(drop=True)
tot_share = sum(e["share"] for e in RUNNABLE)
cur = 0
for j, e in enumerate(RUNNABLE):
    n = (len(block) - cur) if j == len(RUNNABLE) - 1 \
        else int(round(len(block) * e["share"] / tot_share))
    if PAIRING == "shared":
        e["rows"] = block.copy()          # every family sees the whole block
    else:
        e["rows"] = block.iloc[cur:cur + n].reset_index(drop=True)
        cur += n
    e["n"] = len(e["rows"])
for e in SKIPPED:
    e["rows"] = block.iloc[0:0]
    e["n"] = 0

print(f"shard {SHARD}/{N_SHARDS}: reals [{SHARD*BLOCK}:{(SHARD+1)*BLOCK}) "
      f"of the shuffled order, {len(block)} images")
print(f"split across the {len(RUNNABLE)} families this GPU can run\n")
print(f"{'family':28s} {'n':>5s}  first ImageID")
for e in RUNNABLE:
    print(f"{family_of(e):28s} {e['n']:5d}  {e['rows']['image_id'].iloc[0]}")

used = [i for e in RUNNABLE for i in e["rows"]["image_id"]]
if PAIRING == "disjoint":
    assert len(used) == len(set(used)) == len(block), (
        "the block was not partitioned cleanly -- some real is in two families "
        "or none, which breaks 1 real : 1 fake")
    print(f"\n{len(used)} reals, each used exactly once (1 real : 1 fake)")
else:
    print(f"\n{len(block)} reals, each generated by all "
          f"{len(RUNNABLE)} families (1 real : {len(RUNNABLE)} fakes)")

### Prompts, for the arms that take one

§3.2: *do not invent captions*. Open Images ships **Localized Narratives** —
human-written descriptions keyed by `ImageID` — so a `t2i_caption` fake is a
genuine pairing rather than an unrelated photo that happens to be nearby.

Loaded only when a suite entry actually needs it, and only the shards that
cover the selected ids. A row with no narrative is **marked**, not quietly
filled in from the class labels: §3.2 asks for that column and a detector
trained on a mix of the two should be scoreable on the difference.

In [ ]:
NARRATIVE = {}
wants_caption = any(e["method"] == "t2i_caption" for e in SUITE)

if wants_caption and CORPUS == "open_images_v7":
    want_ids = set().union(*[set(e["rows"]["image_id"])
                             for e in SUITE if e["method"] == "t2i_caption"])
    files = sorted({f for root in SEARCH_ROOTS if os.path.isdir(root)
                    for f in glob.glob(os.path.join(root, "**/*localized_narratives*.jsonl"),
                                       recursive=True)})
    if not files:
        base = ("https://storage.googleapis.com/localized-narratives/"
                "annotations/open_images_train_v6_localized_narratives-"
                "{:05d}-of-00010.jsonl")
        os.makedirs(f"{HERE}/ln", exist_ok=True)
        import urllib.request
        for k in range(10):
            dst = f"{HERE}/ln/{k:05d}.jsonl"
            if not os.path.exists(dst):
                print(f"downloading narratives shard {k+1}/10 ...")
                urllib.request.urlretrieve(base.format(k), dst)
            files.append(dst)
            with open(dst) as fh:
                for line in fh:
                    r = json.loads(line)
                    if r["image_id"] in want_ids:
                        NARRATIVE.setdefault(r["image_id"],
                                             " ".join(r["caption"].split()))
            if len(NARRATIVE) >= len(want_ids):
                print("all selected ids covered; stopping early")
                break
    else:
        for p in files:
            with open(p) as fh:
                for line in fh:
                    r = json.loads(line)
                    if r["image_id"] in want_ids:
                        NARRATIVE.setdefault(r["image_id"],
                                             " ".join(r["caption"].split()))
    print(f"narratives for {len(NARRATIVE)}/{len(want_ids)} selected reals "
          f"({100*len(NARRATIVE)/max(1,len(want_ids)):.0f}%)")
elif wants_caption:
    best = {}
    for a in CAPTION_JSON["annotations"]:
        k = str(a["image_id"])
        if k not in best or a["id"] < best[k][0]:
            best[k] = (a["id"], " ".join(a["caption"].split()))
    NARRATIVE = {k: v[1] for k, v in best.items()}
    print(f"COCO captions for {len(NARRATIVE)} images")
else:
    print("no suite entry needs a prompt")

## 6. Geometry and the methods

`generate` is the whole of the generation logic, deliberately in one function:
every method must return a `PIL.Image` at exactly the cropped real's size, from
a seed derived only from the `ImageID`, or the pairing guarantee this notebook
sells is not enforced anywhere.

The seed is `blake2b(SEED, ImageID)` and not a running counter. A counter would
make image *n*'s noise depend on how many images preceded it, so a rerun with a
different `N_TOTAL`, a different shard boundary or one dropped file would
silently produce different pixels for the same real — and "deterministic given
the real" is the property the whole design rests on.

In [ ]:
def crop_box(w, h):
    """Centre box: multiple of 8, long side capped, ASPECT PRESERVED.

    The cap scales both sides by one factor rather than clamping each. Clamping
    each independently would turn a 1200x1800 portrait into a 1024x1024 square
    crop -- silently moving the aspect-ratio distribution of the class that is
    supposed to match its real pair on exactly that. It does not bind on the OV7
    thumbnails (long side ~650) but it binds the moment anyone points this at
    originals, which is when nobody would be watching for it.

    Crops only. Never resizes: a resample leaves a spectral signature and one
    class carrying it is the confound this notebook exists to remove.
    """
    scale = min(1.0, MAX_SIDE / max(w, h))
    cw, ch = int(w * scale), int(h * scale)
    cw, ch = cw - cw % 8, ch - ch % 8
    return ((w - cw) // 2, (h - ch) // 2, (w - cw) // 2 + cw, (h - ch) // 2 + ch)

def seed_for(image_id):
    return int(hashlib.blake2b(f"{SEED}:{image_id}".encode(),
                               digest_size=7).hexdigest(), 16)

def box_mask(image_id, w, h):
    """A deterministic rectangle covering ~25-45% of the frame.

    docs/02 §3.1 wants a partially-synthetic class and OV7's segmentation and
    bbox CSVs are ~2 GB each -- more download than the arm is worth. B-Free used
    plain rectangles for its different-category replacements anyway, so this is
    the honest version of that: it is a BOX, the name says box, and nothing here
    claims the regenerated region follows an object.
    """
    rng = np.random.default_rng(seed_for(image_id) % (2**32))
    fw, fh = rng.uniform(0.5, 0.67, size=2)
    bw, bh = int(w * fw) // 8 * 8, int(h * fh) // 8 * 8
    l, t = int(rng.integers(0, w - bw + 1)), int(rng.integers(0, h - bh + 1))
    m = Image.new("L", (w, h), 0)
    m.paste(255, (l, t, l + bw, t + bh))
    return m

@torch.no_grad()
def vae_roundtrip(pipe, img, dtype):
    x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 127.5 - 1.0)
    x = x.permute(2, 0, 1)[None].to(pipe.vae.device, dtype)
    lat = pipe.vae.encode(x).latent_dist.mode()   # .mode(), not .sample(): this
    out = pipe.vae.decode(lat).sample             # arm isolates the DECODER, and
    out = (out[0].float().permute(1, 2, 0) + 1.0) * 127.5   # a sampled latent
    return Image.fromarray(out.clamp(0, 255).byte().cpu().numpy())  # adds noise

def generate(pipe, spec, method, image_id, real, dtype):
    """real: the cropped PIL real. Returns a PIL fake of identical size."""
    W, H = real.size
    g = torch.Generator("cpu").manual_seed(seed_for(image_id))
    prompt = NARRATIVE.get(image_id, "") if method == "t2i_caption" else ""

    if method == "vae_recon":
        return vae_roundtrip(pipe, real, dtype)

    steps, guid = spec["steps"], spec["guidance"]
    if prompt == "" and guid is not None and guid > 1.0 and method != "ref_image":
        # Classifier-free guidance interpolates a conditional against an
        # unconditional branch. With an empty prompt those are the same tensor,
        # so every guidance value gives the same picture at twice the cost.
        # Dropping to 1.0 makes diffusers skip the second branch outright.
        guid = 1.0
    kw = dict(prompt=prompt, num_inference_steps=steps, generator=g,
              height=H, width=W)
    if guid is not None:
        kw["guidance_scale"] = guid

    if method == "self_cond":
        # The all-zero mask IS the trick. diffusers binarises at 0.5 and forms
        # masked_image = image * (mask < 0.5), so a zero mask hands the UNet the
        # COMPLETE reference latent while telling it nothing is protected: the
        # denoiser starts from pure noise and rebuilds the scene it can see.
        # Every output pixel is generated; none is copied.
        out = pipe(image=real, mask_image=Image.new("L", (W, H), 0),
                   strength=spec["strength"], **kw).images[0]
    elif method == "inpaint_box":
        out = pipe(image=real, mask_image=box_mask(image_id, W, H),
                   strength=spec["strength"], **kw).images[0]
    elif method == "ref_image":
        # FLUX.2 klein has no mask channels; the reference image is a token
        # stream the transformer attends to. Same intent as self_cond, and the
        # only Apache-2.0 way to express it.
        out = pipe(image=real, **kw).images[0]
    elif method == "img2img":
        kw.pop("height"); kw.pop("width")
        out = pipe(image=real, strength=spec["strength"], **kw).images[0]
    elif method == "t2i_caption":
        out = pipe(**kw).images[0]
    else:
        raise AssertionError(method)

    if out.size != (W, H):
        # Some pipelines round to the VAE stride and come back a few pixels
        # large. Cropping is safe (no resampling); resizing is not.
        assert out.size[0] >= W and out.size[1] >= H, f"{out.size} < {(W, H)}"
        out = out.crop((0, 0, W, H))
    return out

## 7. Encoder parity, exactly rather than approximately

`docs/low_level_confounds.md` measured that JPEG history leaks the label, and
§5.2 sets the bar: `jpeg_quality` AUC materially above ~0.60 means fix the save
path before reading any other number.

The usual mitigation is to save fakes "at a matched quality". This is stricter:
the real's **own 64-entry quantisation tables** and its chroma subsampling are
lifted out of its JPEG and the fake is written through them. A per-image match
leaves no distribution difference for the gate to find, because there is no
distribution — every pair shares one encoder configuration.

What it does not fix, and what section 10 measures: the OV7 reals are
`Thumbnail300KURL` re-encodes, so their history is *at least* camera → Flickr →
thumbnail, against the fake's single pass. `FAKE_ENCODE_PASSES = 2` is the lever
if that turns out to be visible, and it is off by default because a second pass
is a guess about a history nobody recorded.

In [ ]:
from PIL import JpegImagePlugin

def source_encoder(path):
    with Image.open(path) as im:
        q = getattr(im, "quantization", None)
        if not q:
            return None
        try:
            ss = JpegImagePlugin.get_sampling(im)
        except Exception:
            ss = -1
        return dict(qtables=q, subsampling=ss if ss in (0, 1, 2) else 2,
                    progressive="progression" in im.info)

def save_matched(img, dst, enc):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if enc is None or not COPY_QTABLES:
        img.save(dst, "JPEG", quality=95, subsampling=2)
        return
    kw = dict(qtables=enc["qtables"], subsampling=enc["subsampling"],
              progressive=enc["progressive"], optimize=False)
    for _ in range(max(1, FAKE_ENCODE_PASSES) - 1):
        buf = io.BytesIO()
        img.save(buf, "JPEG", **kw)
        img = Image.open(buf); img.load()
    img.save(dst, "JPEG", **kw)

# libjpeg's standard table, inverted to report a quality number per row. The
# parity does not depend on this being right -- it is for auditing the manifest
# without reopening every file.
_Q50 = np.array([
    16,11,10,16,24,40,51,61, 12,12,14,19,26,58,60,55, 14,13,16,24,40,57,69,56,
    14,17,22,29,51,87,80,62, 18,22,37,56,68,109,103,77, 24,35,55,64,81,104,113,92,
    49,64,78,87,103,121,120,101, 72,92,95,98,112,100,103,99], dtype=np.float64)

def quality_of(qtables):
    tbl = np.asarray(qtables[0], dtype=np.float64).reshape(64)
    scale = float(np.median(tbl / _Q50)) * 100.0
    q = (5000.0 / scale) if scale > 100.0 else ((200.0 - scale) / 2.0)
    return float(np.clip(q, 1.0, 100.0))

_p = os.path.join(IMG_DIR, POOL["file_name"].iloc[0])
_e = source_encoder(_p)
print("sample real encoder:", "NOT JPEG -- qtable copy will not apply!" if _e is None
      else f"quality~{quality_of(_e['qtables']):.1f}  subsampling={_e['subsampling']}"
           f"  progressive={_e['progressive']}")

## 8. Run the suite

One family at a time, each pipeline loaded, used, and freed before the next —
a 16 GB card holds one of these, not three.

Resumable at row granularity: a `.jsonl` line is appended as each fake lands and
is reloaded on restart. Kaggle's session limits make this non-optional, and it
is only safe because `seed_for` depends on nothing but the `ImageID`.

Every output is checked before it counts. A near-constant frame is a NaN latent
or a safety checker's black image; a frame *identical to its real* means the
pipeline composited the original pixels back in, and the family would be a
directory of copies labelled "fake". Both raise.

In [ ]:
from diffusers import (AutoPipelineForInpainting, AutoPipelineForImage2Image,
                       AutoPipelineForText2Image)
import diffusers as _dfs

AUTO = {"inpaint": AutoPipelineForInpainting,
        "img2img": AutoPipelineForImage2Image, "t2i": AutoPipelineForText2Image}

def check_licence(spec):
    """Ask the Hub whether the registry's licence claim is still true.

    The registry records what I believe; this asks whether it holds. A licence
    changed upstream, or one recorded wrongly in the first place, stops the run
    here rather than surfacing in an audit of a finished corpus -- which is the
    failure docs/02 §2 is guarding against when it says to verify every time.
    """
    from huggingface_hub import model_info
    from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
    try:
        tag = (model_info(spec["hf_id"], token=HF_TOKEN).cardData or {}).get("license")
    except RepositoryNotFoundError:
        raise SystemExit(f"{spec['hf_id']} is not on the Hub. Fix the registry "
                         "entry rather than guessing a near-miss id.")
    except GatedRepoError:
        raise SystemExit(
            f"{spec['hf_id']} is GATED and the token cannot see it. Accept its "
            "terms on the model page while logged in -- and read the licence "
            "while you are there; a gated repo is gated because someone wants "
            "you to agree to something.")
    except Exception as exc:
        print(f"  licence check skipped ({type(exc).__name__}); "
              f"registry says {spec['licence']}")
        return
    want = spec.get("licence_tag")
    print(f"  hub licence tag: {tag!r}")
    assert want is None or tag is None or tag == want, (
        f"{spec['hf_id']} is published as {tag!r} but this registry expects "
        f"{want!r}. Read the model page before generating anything. Update the "
        "registry only after reading it -- never to make this line pass.")

def load(model_key, method):
    spec = MODELS[model_key]
    check_licence(spec)
    dtype = torch.bfloat16 if spec.get("needs_bf16") else torch.float16
    kind = METHODS[method][0]
    cls = AUTO[kind]
    if spec.get("pipe_class"):
        cls = getattr(_dfs, spec["pipe_class"], cls)
    kw = dict(torch_dtype=dtype, use_safetensors=True)
    if spec["lineage"] == "sd_vae":
        # Only SD 1.x/2.x have this component. It returns a BLACK IMAGE on a
        # positive, and a black frame entering a forensics corpus as a "fake" is
        # a labelled row that teaches nothing -- the pixel check below rejects degenerate
        # outputs on their pixels instead, which also catches NaN latents.
        kw.update(safety_checker=None, requires_safety_checker=False)
    if HF_TOKEN:
        kw["token"] = HF_TOKEN
    try:
        pipe = cls.from_pretrained(spec["hf_id"], variant="fp16", **kw)
    except Exception:
        pipe = cls.from_pretrained(spec["hf_id"], **kw)
    pipe.set_progress_bar_config(disable=True)
    if spec["vram_gb"] > VRAM:
        pipe.enable_sequential_cpu_offload()
    else:
        pipe = pipe.to("cuda")
    for fn in ("enable_vae_slicing", "enable_attention_slicing"):
        try:
            getattr(pipe, fn)()
        except Exception:
            pass
    if kind == "inpaint":
        ch = pipe.unet.config.in_channels
        assert ch == 9, (
            f"{spec['hf_id']} has a {ch}-channel UNet, not 9. self_cond depends "
            "on the 5 extra channels -- 1 mask + 4 masked-image latent -- being "
            "how the reference image reaches the denoiser. On a 4-channel UNet "
            "diffusers emulates inpainting by BLENDING the original latents back "
            "each step, which with an all-zero mask returns the input unchanged: "
            "a family of perfect copies labelled 'fake'.")
    return pipe, dtype

def check(fake, real, image_id, method):
    a = np.asarray(fake, dtype=np.float32)
    if a.std() < 2.0:
        raise RuntimeError(
            f"{image_id}: output is near-constant (std={a.std():.2f}) -- a NaN "
            "latent or a safety-checker black frame, not a picture.")
    d = float(np.abs(a - np.asarray(real, dtype=np.float32)).mean())
    if method != "vae_recon" and d < 1.5:
        raise RuntimeError(
            f"{image_id}: the output is {d:.2f}/255 from its input, i.e. the "
            "same picture. The pipeline composited the original pixels back "
            "over the generated ones. Writing this would fill the family with "
            "copies labelled 'fake'.")
    return d

os.makedirs(f"{OUT_ROOT}/{SOURCE}/real", exist_ok=True)
ALL_ROWS = []

# RUNNABLE, not SUITE: section 5 already partitioned this shard's block across
# exactly these families, so every row here has reals waiting and every real in
# the block has a family.
for e in RUNNABLE:
    fam, spec, method = family_of(e), MODELS[e["model"]], e["method"]
    jl = f"{OUT_ROOT}/rows_{fam}_s{SHARD}of{N_SHARDS}.jsonl"
    done = set()
    if os.path.exists(jl):
        with open(jl) as f:
            for line in f:
                try:
                    done.add(json.loads(line)["image_id"])
                except Exception:
                    pass
    todo = e["rows"][~e["rows"]["image_id"].isin(done)]
    print(f"\n===== {fam} " + "=" * (46 - len(fam)))
    print(f"{spec['hf_id']}  |  {len(done)} done, {len(todo)} to go")
    if todo.empty:
        continue

    t0 = time.time()
    pipe, dtype = load(e["model"], method)
    print(f"loaded {type(pipe).__name__} in {time.time()-t0:.0f}s")

    t0, made, errs = time.time(), 0, []
    for k, (_, r) in enumerate(todo.iterrows(), 1):
        iid = r["image_id"]
        src = os.path.join(IMG_DIR, r["file_name"])
        try:
            with Image.open(src) as im:
                im = im.convert("RGB")
                real = im.crop(crop_box(*im.size))
            enc = source_encoder(src)
            ts = time.time()
            fake = generate(pipe, spec, method, iid, real, dtype)
            gen_s = time.time() - ts
            delta = check(fake, real, iid, method)
            save_matched(fake, f"{OUT_ROOT}/{SOURCE}/{fam}/{iid}.jpg", enc)
            if EMIT_REALS:
                # Byte-for-byte. Re-encoding to "match" would add a compression
                # generation to the authentic class -- the exact confound
                # section 7 exists to remove.
                shutil.copyfile(src, f"{OUT_ROOT}/{SOURCE}/real/{iid}.jpg")
            row = dict(
                image_id=iid, source=SOURCE, bucket=fam, label=1,
                rel_fake=f"{SOURCE}/{fam}/{iid}.jpg",
                rel_real=f"{SOURCE}/real/{iid}.jpg" if EMIT_REALS else "",
                corpus=CORPUS, model=e["model"], hf_id=spec["hf_id"],
                weight_licence=spec["licence"], lineage=spec["lineage"],
                method=method, synthesis="partial" if method == "inpaint_box" else "full",
                prompt=NARRATIVE.get(iid, "") if method == "t2i_caption" else "",
                prompt_source=("localized_narrative"
                               if method == "t2i_caption" and iid in NARRATIVE
                               else "none" if method != "t2i_caption" else "MISSING"),
                steps=spec["steps"], guidance=spec["guidance"],
                strength=spec["strength"], seed=seed_for(iid),
                width=fake.size[0], height=fake.size[1],
                real_width=int(r["width"]), real_height=int(r["height"]),
                jpeg_quality=round(quality_of(enc["qtables"]), 2) if enc else None,
                qtables_copied=bool(enc and COPY_QTABLES),
                encode_passes=FAKE_ENCODE_PASSES,
                mean_abs_delta=round(delta, 3), gen_seconds=round(gen_s, 2),
                image_licence=r["licence"], author=r["author"],
                original_url=r["original_url"],
            )
            with open(jl, "a") as f:
                f.write(json.dumps(row) + "\n")
            ALL_ROWS.append(row); made += 1
        except Exception as exc:                       # noqa: BLE001
            errs.append((iid, repr(exc)))
            if len(errs) > max(3, 0.05 * len(todo)):
                raise RuntimeError(
                    f"{len(errs)} failures in {fam}, last {errs[-1]}. This is "
                    "the run, not individual images.") from exc
        if k % 10 == 0 or k == len(todo):
            el = time.time() - t0
            print(f"  {k:5d}/{len(todo)}  {el/max(1,made):5.2f}s/img  "
                  f"eta {(len(todo)-k)*el/max(1,made)/60:6.1f}m")
    if errs:
        print(f"  {len(errs)} failed, e.g. {errs[0]}")
    print(f"  {made} written, {(time.time()-t0)/max(1,made):.2f}s/img")

    del pipe
    torch.cuda.empty_cache()

print(f"\n{len(ALL_ROWS)} fakes written this session")

## 9. Look at them — this is the "physically verified" step

Real on top, its fake below, one column per family. Read it properly before
setting `SMOKE = False`; nothing downstream can tell you what this cell can.

* **`self_cond` / `ref_image`** should be *recognisably the same scene* — same
  layout, same palette, same object count — while being visibly redrawn. A
  pixel-perfect copy should already have raised in section 8. A completely
  different scene means `strength` is wrong for that checkpoint or the mask is
  not arriving as all-zeros.
* **`inpaint_box`** should show one rectangle of new content in an otherwise
  authentic frame. If the whole frame changed, the mask is being ignored.
* **`t2i_caption`** shares only the subject. That is correct and is the point of
  having it.
* **`vae_recon`** should be almost indistinguishable by eye. Also correct: it is
  a hard negative, and Δ in the single digits is what it is for.

In [ ]:
import matplotlib.pyplot as plt

by_fam = {}
for r in ALL_ROWS:
    by_fam.setdefault(r["bucket"], r)
if by_fam:
    fams = list(by_fam)
    fig, ax = plt.subplots(2, len(fams), figsize=(3.3 * len(fams), 7.2))
    ax = np.atleast_2d(ax).reshape(2, -1)
    for j, fam in enumerate(fams):
        r = by_fam[fam]
        src = os.path.join(IMG_DIR, FILE_NAME[r["image_id"]])
        with Image.open(src) as im:
            real = im.convert("RGB").crop(crop_box(*im.size))
        fake = Image.open(f"{OUT_ROOT}/{r['rel_fake']}")
        ax[0, j].imshow(real); ax[0, j].set_title("real", fontsize=9)
        ax[1, j].imshow(fake)
        ax[1, j].set_title(f"{fam}\nΔ={r['mean_abs_delta']:.0f}/255  "
                           f"{r['gen_seconds']:.1f}s", fontsize=8)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    plt.tight_layout(); plt.show()
    for fam, r in by_fam.items():
        print(f"{fam:28s} prompt={r['prompt'][:60]!r}")
else:
    print("nothing generated this session (everything resumed from disk)")

## 10. The gate — run it on 2,000 before generating 58,000

§4 says stop at 2,000 and check; §5 says the check is allowed to cancel the
task. This is that check, computed inline against the frozen corpus baselines.

| proxy | baseline | a high value means |
| --- | --- | --- |
| `jpeg_quality` | 0.5532 | separable on compression alone |
| `laplacian_var` | 0.6721 | separable on sharpness |
| `noise_floor` | 0.6374 | separable on sensor/denoiser noise |

Two things make this stricter here than the same numbers on the frozen corpus.
Content is matched image-for-image, so a proxy cannot pass by accidentally
tracking subject matter — it has nothing left to find but the encoder and the
generator. And it is reported **per family**, because a `vae_recon` family and a
`t2i_caption` family fail for different reasons and averaging them hides both.

The AUC is folded around 0.5 (`|auc - 0.5| + 0.5`) before comparison, because a
proxy that ranks fakes *below* reals is exactly as much of a shortcut as one
that ranks them above.

In [ ]:
import cv2

PROXY_SOURCE = "ported (repo unavailable)"
if HAVE_REPO:
    try:
        from aigcdet.features.proxies import (estimate_jpeg_quality,
                                              laplacian_variance, noise_floor)
        laplacian_var = laplacian_variance
        def jpeg_q(path):
            with Image.open(path) as im:
                return estimate_jpeg_quality(np.asarray(im.convert("RGB")), path)
        PROXY_SOURCE = "aigcdet.features.proxies"
    except Exception as exc:
        print(f"repo present but proxies did not import ({exc}); using the port")

if PROXY_SOURCE != "aigcdet.features.proxies":
    def _grey(a): return cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    def laplacian_var(a): return float(cv2.Laplacian(_grey(a), cv2.CV_64F).var())
    def noise_floor(a):
        g = _grey(a).astype(np.float32)
        r = np.abs((g - cv2.GaussianBlur(g, (0, 0), 1.0)))
        return float(np.median(np.abs(r - np.median(r))) * 1.4826)
    def jpeg_q(p):
        with Image.open(p) as im:
            q = getattr(im, "quantization", None)
        return quality_of(q) if q else float("nan")

print(f"proxies: {PROXY_SOURCE}")
if PROXY_SOURCE != "aigcdet.features.proxies":
    print("  NOTE: the 0.5532 / 0.6721 / 0.6374 baselines were produced by the "
          "repo's estimators.\n  These are equivalent in intent but not "
          "guaranteed identical in value -- treat a near-baseline\n  result as "
          "inconclusive and re-run with the repo available.")

def auc(pos, neg):
    pos = np.asarray(pos, float); neg = np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if not len(pos) or not len(neg):
        return float("nan")
    rank = np.concatenate([pos, neg]).argsort().argsort() + 1
    return float((rank[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2)
                 / (len(pos) * len(neg)))

BASELINE = {"jpeg_quality": 0.5532, "laplacian_var": 0.6721, "noise_floor": 0.6374}
NAMES = FILE_NAME

rows_all = []
for e in SUITE:
    jl = f"{OUT_ROOT}/rows_{family_of(e)}_s{SHARD}of{N_SHARDS}.jsonl"
    if os.path.exists(jl):
        rows_all += [json.loads(l) for l in open(jl)]

print(f"{'family':28s} {'n':>4s} {'jpeg_q':>8s} {'laplac':>8s} {'noise':>8s}  verdict")
GATE = {}
for e in SUITE:
    fam = family_of(e)
    rs = [r for r in rows_all if r["bucket"] == fam][:400]
    if not rs:
        continue
    P = {k: ([], []) for k in BASELINE}
    for r in rs:
        fp = f"{OUT_ROOT}/{r['rel_fake']}"
        rp = os.path.join(IMG_DIR, NAMES.get(r["image_id"], f"{r['image_id']}.jpg"))
        if not (os.path.exists(fp) and os.path.exists(rp)):
            continue
        with Image.open(rp) as im:
            ra = np.asarray(im.convert("RGB").crop(crop_box(*im.size)))
        fa = np.asarray(Image.open(fp).convert("RGB"))
        P["jpeg_quality"][0].append(jpeg_q(fp)); P["jpeg_quality"][1].append(jpeg_q(rp))
        P["laplacian_var"][0].append(laplacian_var(fa)); P["laplacian_var"][1].append(laplacian_var(ra))
        P["noise_floor"][0].append(noise_floor(fa));     P["noise_floor"][1].append(noise_floor(ra))
    a = {k: auc(*v) for k, v in P.items()}
    folded = {k: abs(v - 0.5) + 0.5 for k, v in a.items()}
    bad = [k for k in BASELINE if folded[k] > BASELINE[k] + 0.03]
    GATE[fam] = dict(auc=a, folded=folded, fail=bad, n=len(rs))
    print(f"{fam:28s} {len(rs):4d} {a['jpeg_quality']:8.4f} "
          f"{a['laplacian_var']:8.4f} {a['noise_floor']:8.4f}  "
          f"{'FAIL: ' + ','.join(bad) if bad else 'ok'}")

jq_fail = [f for f, g in GATE.items() if g["folded"]["jpeg_quality"] > 0.60]
other = [f for f, g in GATE.items() if set(g["fail"]) - {"jpeg_quality"}]
print()
if jq_fail:
    print("STOP -- compression separates the classes in:", jq_fail)
    print("Fix the SAVE PATH and nothing else (§5.2):")
    print("  - did source_encoder find tables? a PNG copy of the reals has none")
    print("  - try FAKE_ENCODE_PASSES = 2: the OV7 reals are Thumbnail300KURL")
    print("    re-encodes and already carry a generation the fakes do not")
elif other:
    print("A non-compression proxy is above baseline in:", other)
    print("That is a REAL FINDING (§6), not a bug. With content, geometry and")
    print("encoder matched pair-for-pair, a sharpness or noise gap is a property")
    print("of the GENERATOR. Write it up, and note which family: laplacian_var")
    print("failing on a 4-step model is a different story from noise_floor")
    print("failing on a 50-step one.")
elif GATE:
    print("All families at or below baseline on all three proxies. Scale.")

## 11. Freeze the outputs

Three files. The third is not optional: CC BY *requires* attribution,
normalisation strips image metadata, and this is then the only surviving record
of who took the photograph. It ships with the dataset or the dataset is not
licence-clean.

In [ ]:
df = pd.DataFrame(rows_all).drop_duplicates(["bucket", "image_id"], keep="last")
pairs = f"{OUT_ROOT}/pairs_s{SHARD}of{N_SHARDS}.parquet"
df.to_parquet(pairs, index=False)

att_out = (df[["image_id", "author", "original_url", "image_licence"]]
           .drop_duplicates("image_id")
           .rename(columns={"image_id": "ImageID", "author": "Author",
                            "original_url": "OriginalURL",
                            "image_licence": "License"}))
ap = f"{OUT_ROOT}/attribution.csv"
if os.path.exists(ap):
    att_out = pd.concat([pd.read_csv(ap), att_out]).drop_duplicates("ImageID")
att_out.to_csv(ap, index=False)
blank = att_out[att_out[["Author", "OriginalURL"]].isna().any(axis=1)
                | att_out[["Author", "OriginalURL"]].eq("").any(axis=1)]
assert blank.empty, f"{len(blank)} attribution rows have blanks (§5.4)"

json.dump(dict(
    corpus=CORPUS, source=SOURCE, pairing=PAIRING, seed=SEED,
    shard=SHARD, n_shards=N_SHARDS, smoke=SMOKE,
    min_side=MIN_SIDE, max_side=MAX_SIDE,
    copy_qtables=COPY_QTABLES, encode_passes=FAKE_ENCODE_PASSES,
    gpu=GPU, sm=f"{CAP[0]}{CAP[1]}",
    families={family_of(e): dict(
        model=e["model"], hf_id=MODELS[e["model"]]["hf_id"],
        licence=MODELS[e["model"]]["licence"],
        lineage=MODELS[e["model"]]["lineage"], method=e["method"],
        n=int((df["bucket"] == family_of(e)).sum()),
        gate=GATE.get(family_of(e), {}).get("folded")) for e in SUITE},
), open(f"{OUT_ROOT}/config_s{SHARD}of{N_SHARDS}.json", "w"), indent=2)

print(df.groupby(["lineage", "bucket"]).size().to_string())
print(f"\n{len(df)} pairs -> {pairs}")
print(f"{len(att_out)} attribution rows -> {ap}")
print(f"reals emitted: {len(glob.glob(f'{OUT_ROOT}/{SOURCE}/real/*.jpg'))}")
print("\nSave Version > Save & Run All, then publish /kaggle/working/aigc_pairs "
      "as a Dataset. Attach it to the build_dataset run as --raw.")

## 12. The shard plan — how many images fit in an hour, and who runs what

`TARGET_SHARD_HOURS` is what one session of this notebook should cost. This
cell turns it into the `N_TOTAL` / `N_SHARDS` that hits it, using **measured**
throughput where this session produced any and the prior table where it did not.

**How to run this with other people.** Everyone opens the same notebook with the
same `SUITE`, the same `SEED`, and the same `N_TOTAL` and `N_SHARDS`; each
person changes only `SHARD`. Shard 3 of 12 draws the same reals on every
machine, because the slices in section 6 are cut over the **full declared
suite** — before any capability filtering — and indexed by `SHARD`. So a friend
on a 4090 and you on Kaggle can hold different `SHARD` values and never collide,
and the 4090 additionally fills in the bf16 families the T4 skipped. There is no
coordination beyond those four numbers.

The one rule: **`N_SHARDS` must be identical everywhere.** It is the divisor
that defines what a shard *is*. Change it on one machine and that machine's
shard 3 is a different set of reals from everyone else's shard 3 — which shows
up not as an error but as some reals generated twice and others never.

In [ ]:
measured = {}
for e in SUITE:
    jl = f"{OUT_ROOT}/rows_{family_of(e)}_s{SHARD}of{N_SHARDS}.jsonl"
    if os.path.exists(jl):
        secs = [json.loads(l).get("gen_seconds", 0) for l in open(jl)]
        secs = [x for x in secs if x]
        if secs:
            measured[family_of(e)] = float(np.median(secs))

def rate_of(e):
    fam = family_of(e)
    if fam in measured:
        return measured[fam], "measured"
    prior = RATE_PRIOR_T4.get((e["model"], e["method"]))
    if prior is None:
        return None, "unknown"
    return prior * DEV_FACTOR, f"prior x{DEV_FACTOR:.2f}"

print(f"{GPU}  target {TARGET_SHARD_HOURS:.1f} h per shard\n")
print(f"{'family':28s} {'s/img':>8s} {'source':>14s} {'img/hour':>9s} {'runs here':>10s}")
total_s = 0.0
for e in SUITE:
    r, how = rate_of(e)
    here = "no (bf16)" if e in SKIPPED else "yes"
    if r is None:
        print(f"{family_of(e):28s} {'?':>8s} {how:>14s} {'?':>9s} {here:>10s}")
        continue
    total_s += r * e["n"]
    print(f"{family_of(e):28s} {r:8.2f} {how:>14s} {3600/r:9.0f} {here:>10s}")

if total_s:
    budget = TARGET_SHARD_HOURS * 3600
    want = max(1, int(np.ceil(total_s / budget)))
    print(f"\nwhole suite at N_TOTAL={sum(e['n'] for e in SUITE)}: "
          f"{total_s/3600:.1f} GPU-hours on this device")
    print(f"-> N_SHARDS = {want} puts each shard at ~{total_s/want/3600:.2f} h")
    if SMOKE:
        print("\n(SMOKE is on, so N_TOTAL is 3-per-family. Re-read this cell "
              "after the first real run: the measured column is what you want "
              "to plan from, and it only exists for families this box ran.)")
    else:
        print(f"\nSet N_SHARDS = {want} in section 1 on EVERY machine, hand "
              f"out SHARD = 0..{want-1}, and go.")

# Persist the measured rates so the next session on this box plans from them.
if measured:
    rp = f"{OUT_ROOT}/rates_{GPU.replace(' ', '_')}.json"
    old = json.load(open(rp)) if os.path.exists(rp) else {}
    old.update(measured)
    json.dump(old, open(rp, "w"), indent=2)
    print(f"\nmeasured rates -> {rp}")

## 13. Wiring it into the corpus

**Register the source.** `src/aigcdet/data/sources.py`:

```python
"open_images_v7": SourceSpec(
    name="open_images_v7",
    licence="CC BY 2.0 (per-image) — https://creativecommons.org/licenses/by/2.0/ "
            "— attribution required; ship attribution.csv",
    real_buckets=frozenset({"real"}),
    generator_buckets=True,
    exclude_from_training=False,
),
```

The layout written above is already `<source>/<bucket>/...` with the family at
`rel[1]`, which is the depth `build_dataset.py` reads the bucket from.

**Group held-out by decoder, not by name** (§3.4). The `lineage` column exists
so this is a groupby rather than someone's memory:

```python
heldout_groups = [
    ["sd15_self_cond", "sd15_inpaint_box",
     "sd21_self_cond", "sd21_inpaint_box", "sd21_vae_recon"],   # sd_vae
    ["sdxl_self_cond", "sdxl_inpaint_box"],                     # sdxl_vae
    ["flux_schnell_t2i_caption"],                               # flux_vae
]
```

Holding out `sdxl_*` while `sd21_*` trains measures generalising to a *cousin* —
retrained VAE, same architecture. Holding out the FLUX group measures a real
lineage jump. Both are worth a number; they are not the same number, and the
current split's `SDwithAdaptor_controlnet` / `VQGAN` choice is the first kind
being read as the second.

**Split by `image_id`, not by row.** Every fake carries the `ImageID` it came
from. If a real and its fake land on opposite sides of the train/val boundary
the model sees one scene under both labels and validation is optimistic. Under
`PAIRING="shared"` this is not a subtlety — it is six rows of the same scene.

---

### The residual, stated plainly

Content, geometry and final encoder are matched pair-for-pair. Three things are
not, and pretending otherwise would be worse than the confound:

* **Compression history.** The reals are `Thumbnail300KURL` re-encodes carrying
  at least camera → Flickr → thumbnail; the fakes carry one pass through the
  reals' own tables. Section 10 measures whether that is visible;
  `FAKE_ENCODE_PASSES` is the lever.
* **Aspect and scale.** The harvest filtered to AR ≤ 0.7 and short side ≥ 400,
  so this corpus is portrait and roughly one scale. `docs/resolution_shortcut.md`
  applies unchanged, and a detector trained only here has seen one geometry.
* **`self_cond` is not what an adversary does.** It is image-conditioned
  regeneration; a real attacker prompts from scratch. That is why
  `t2i_caption` holds 20% rather than 0 — `self_cond` isolates the artifact,
  `t2i_caption` resembles the threat, and a corpus wants both. Score them
  separately: `synthesis` and `method` are on every row for exactly that.

### If the gate fails and the save path is not the cause

That is §6's negative result and it is publishable, not a bug to route around.
It would mean generated and photographic images differ, on this source, at a
level canonicalisation cannot remove — which is worth knowing before a detector
is built on the assumption that it can.

---

# HANDOFF — the decisions behind this notebook

For a session picking this up cold. Each item is a decision that was made for a
reason; changing one without the reason is how this stops working.

## Why generate at all
Every fake in the corpus today is an *unrelated* picture beside its reals, so
camera, encoder, aspect and subject all differ with the label
(`docs/low_level_confounds.md`). Balancing counts does not fix that; holding
content fixed does. Detection accuracy also falls ~79% → ~38% from 2020-21 to
2024 generators, and every public dataset that would close that gap is
licence-barred. Hence: generate our own, over reals we may redistribute.

## Why `self_cond` rather than prompting
An inpainting UNet takes 9 channels: 4 noisy latent, **1 mask**, **4 masked-image
latent**. An all-zero mask says "nothing is protected", so the reference image
arrives in full and the denoiser's task becomes *regenerate this*. No caption.
B-Free (CVPR 2025) built 309k fakes over 51k reals this way for this exact bias
reason; DRCT (ICML 2024) and TwinSynths (WACV 2025) are the same idea.
`ref_image` is the same intent on FLUX.2 klein, which has no mask channels.
`t2i_caption` is kept at ~20% because `self_cond` isolates the artifact while
`t2i_caption` resembles what an adversary actually does; a corpus wants both.

## Why the JPEG tables are copied, not matched
The reals are `Thumbnail300KURL` **re-encodes**. Saving fakes at "a similar
quality" leaves a distribution the gate can find. Lifting the real's own 64
integers and subsampling onto its fake leaves no distribution at all. Measured:
`jpeg_quality` AUC **0.5031** against a 0.5532 baseline. Never re-encode a real
to "match" — that adds a compression generation to the authentic class.

## Why nothing is ever resized
A resample leaves a spectral signature; one class carrying it is the confound
(`docs/resolution_shortcut.md`). Geometry is centre-**cropped** to a multiple of
8, aspect preserved. `MAX_SIDE` scales both sides by one factor — clamping each
independently would square up a portrait and move the aspect distribution.

## Licences — the two traps found here
* **SDXL-Turbo** is `sai-nc-community`, **non-commercial**. `docs/02` §2 lists it
  as OpenRAIL++-M "Use." That is wrong and the doc still needs fixing.
* **FLUX.2-klein-9B is non-commercial**; only **4B** is Apache-2.0. Same trap as
  FLUX.1-dev. Reaching for the bigger sibling is the obvious move and it is
  barred. The 9B entry is kept in the registry *refused*, so the refusal reads
  as a decision rather than an oversight.

Every entry carries `commercial`; the suite validator refuses a non-commercial
model, and `load()` asks the Hub for the published licence tag and asserts it
matches the registry. A licence recorded wrongly stops the run rather than
surfacing in an audit of a finished corpus.

## Why bf16 is refused rather than worked around
fp16 has 5 exponent bits to bf16's 8. A flow-matching DiT in fp16 does not
raise — it emits washed-out or NaN-speckled images that still look like images,
which is the worst failure for a corpus recording what a generator's output
*looks like*. Quantising is refused for a related reason: a 4-bit model's traces
are partly your compute budget's, and the corpus would record both. Kaggle has
only T4 (sm_75) and P100 (sm_60); neither has bf16.

## Known residuals — do not paper over these
* **Compression history.** Reals carry camera → Flickr → thumbnail; fakes carry
  one pass. `FAKE_ENCODE_PASSES = 2` is the lever if section 10 says it shows.
* **One geometry.** The harvest filtered to AR ≤ 0.7, short side ≥ 400. A
  detector trained only here has seen one scale.
* **Per-family counts are a target, not a guarantee.** Rebalancing inside a
  shard block means counts depend on the fleet's GPU mix. Section 11 prints
  what was achieved; top up a thin family with `FAMILIES`.

## Downstream
Register `open_images_v7` in `sources.py` (§13). Group `heldout_groups` by the
`lineage` column — the DECODER — not by name: `sd_vae`, `sdxl_vae`, `flux_vae`,
`flux2_vae`. Holding out `sdxl_*` while `sd21_*` trains measures a *cousin*;
holding out FLUX measures a lineage jump. Both are worth a number and they are
not the same number. **Split train/val by `image_id`**, never by row.

## If the gate fails and the save path is not the cause
That is §6's negative result: generated and photographic images differ, on this
source, below what canonicalisation can remove. Write it up. It is a finding,
not a bug to route around.